In [50]:
from datetime import datetime, timezone
from googleapiclient.discovery import build
from google.oauth2 import service_account
from googleapiclient.http import MediaFileUpload
from google.cloud import storage
import geopandas as gpd
import gspread
from gspread_dataframe import set_with_dataframe

In [43]:
SERVICE_ACCOUNT_FILE = "env/gcs_access.json"

## Load data into Google Drive

### Setup service account access

In [44]:
SCOPES = [
  "https://www.googleapis.com/auth/spreadsheets",
  "https://www.googleapis.com/auth/drive"
]

In [45]:
credentials = service_account.Credentials.from_service_account_file(
  SERVICE_ACCOUNT_FILE, scopes=SCOPES
)

In [46]:
drive_service = build("drive", "v3", credentials=credentials)

### Working with specific google drive folder

In [6]:
folder_id = "1hLnsP_-OFyb6lQkQzzJ-Z1a5qXH6cz-8"

file_metadata = {
  "name": "example.txt",
  "parents": [folder_id],
}

media = MediaFileUpload("example.txt", mimetype="text/plain")
file = drive_service.files().create(
  body=file_metadata, media_body=media, fields="id"
).execute()

print(f"File uploaded to shared folder. File ID: {file['id']}")

File uploaded to shared folder. File ID: 17PgqohCZZ2NK3ALLvaKgmxWbu3CGO7xH


In [14]:
def list_files_in_folder(folder_id):
  """Lists all files in a specific Google Drive folder."""
  query = f"'{folder_id}' in parents and trashed=false"

  results = drive_service.files().list(q=query, fields="files(id, name, mimeType)").execute()
  files = results.get("files", [])

  if not files:
    print("No files found in the folder.")
    return

  print("Files in folder:")
  for file in files:
    print(file)

In [19]:
list_files_in_folder(folder_id)

Files in folder:
{'mimeType': 'text/plain', 'id': '17PgqohCZZ2NK3ALLvaKgmxWbu3CGO7xH', 'name': 'example.txt'}


### Upload geospatial data to google drive

In [6]:
file_path_gpkg = "../../transform/output.gpkg"

In [7]:
def now(extension=None):
  timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M-%S")
  if extension is None:
    return timestamp
  else:
    return timestamp + "." + extension

In [31]:
now()

'2025-03-09_09-18-12'

In [33]:
now("gpkg")

'2025-03-09_09-18-20.gpkg'

In [36]:
folder_id = "1hLnsP_-OFyb6lQkQzzJ-Z1a5qXH6cz-8"

file_metadata = {
  "name": now("gpkg"),
  "parents": [folder_id],
}

media = MediaFileUpload(file_path_gpkg, mimetype="application/geopackage+sqlite3")
file = drive_service.files().create(
  body=file_metadata, media_body=media, fields="id"
).execute()

print(f"File uploaded to shared folder. File ID: {file['id']}")

File uploaded to shared folder. File ID: 1Kukdc69DApH9gSrrJXUn5e9Xxc3u_tTv


In [8]:
file_path_tiff = "../../transform/georeferenced.tif"

In [58]:
folder_id = "1hLnsP_-OFyb6lQkQzzJ-Z1a5qXH6cz-8"

file_metadata = {
  "name": now("tif"),
  "parents": [folder_id],
}

media = MediaFileUpload(file_path_tiff, mimetype="image/tiff", resumable=True)
file = drive_service.files().create(
  body=file_metadata, media_body=media, fields="id"
).execute()

print(f"File uploaded to shared folder. File ID: {file['id']}")

File uploaded to shared folder. File ID: 1Y3eT9LfrWQInwgPwZ0JGnKnv-JPyzLjZ


In [59]:
list_files_in_folder(folder_id)

Files in folder:
{'mimeType': 'image/tiff', 'id': '1Y3eT9LfrWQInwgPwZ0JGnKnv-JPyzLjZ', 'name': '2025-03-09_09-37-44.tif'}
{'mimeType': 'application/geopackage+sqlite3', 'id': '1Kukdc69DApH9gSrrJXUn5e9Xxc3u_tTv', 'name': '2025-03-09_09-20-20.gpkg'}
{'mimeType': 'text/plain', 'id': '17PgqohCZZ2NK3ALLvaKgmxWbu3CGO7xH', 'name': 'example.txt'}


### Upload data in google sheets

In [28]:
df = gpd.read_file(file_path_gpkg, layer="airports_indonesia")

df.head()

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,country_name,iso_country,...,icao_code,iata_code,local_code,home_link,wikipedia_link,keywords,score,last_updated,source,geometry
0,26751,WADD,large_airport,Denpasar I Gusti Ngurah Rai International Airport,-8.748409,115.167123,14.0,AS,Indonesia,ID,...,WADD,DPS,None,http://www.angkasapura1.co.id/eng/location/bal...,https://en.wikipedia.org/wiki/Ngurah_Rai_Inter...,"WRRR, Bali, Denpasar International Airport, De...",1200,2025-02-21T20:46:40+00:00,OurAirports,POINT Z (115.16712 -8.74841 14)
1,26835,WIII,large_airport,Soekarno-Hatta International Airport,-6.125570,106.655998,34.0,AS,Indonesia,ID,...,WIII,CGK,None,http://www.jakartasoekarnohattaairport.com/,https://en.wikipedia.org/wiki/Soekarno-Hatta_I...,"JKT, Cengkareng, Java",51600,2013-02-07T12:57:41+00:00,OurAirports,POINT Z (106.656 -6.12557 34)
2,26787,WARJ,medium_airport,Adisutjipto International Airport,-7.788180,110.431999,379.0,AS,Indonesia,ID,...,None,JOG,None,http://adisutjipto-airport.co.id/,https://en.wikipedia.org/wiki/Adisutjipto_Inte...,"WIIJ, WARJ, Adisucupto",500,2022-10-01T23:14:17+00:00,OurAirports,POINT Z (110.432 -7.78818 379)
3,26789,WARR,large_airport,Juanda International Airport,-7.379830,112.787003,9.0,AS,Indonesia,ID,...,WARR,SUB,None,http://www.juanda-airport.com/,https://en.wikipedia.org/wiki/Juanda_Internati...,WRSJ,51150,2022-10-01T23:14:37+00:00,OurAirports,POINT Z (112.787 -7.37983 9)
4,299629,WADL,medium_airport,Lombok International Airport,-8.759962,116.278169,319.0,AS,Indonesia,ID,...,WADL,LOP,None,http://www.lombok-airport.co.id/,https://en.wikipedia.org/wiki/Lombok_Internati...,"Zainuddin Abdul Madjid, Praya",500,2025-02-21T23:53:28+00:00,OurAirports,POINT Z (116.27817 -8.75996 319)


In [47]:
gc = gspread.authorize(credentials)
gc

In [48]:
def create_spreadsheet_in_folder(sheet_name, folder_id):
  sheet = gc.create(sheet_name)
  file_id = sheet.id
  
  drive_service.files().update(
    fileId=file_id,
    addParents=folder_id,
    removeParents="root",
    fields="id, parents"
  ).execute()

  print(f"Spreadsheet created: {sheet.url}")
  return sheet

In [49]:
create_spreadsheet_in_folder("My GIS Data", folder_id)

Spreadsheet created: https://docs.google.com/spreadsheets/d/1YNkS6sLfZ4-L9TSaMLZxxmUKa-Xtrkt6mUwGjNYIzE0


<Spreadsheet 'My GIS Data' id:1YNkS6sLfZ4-L9TSaMLZxxmUKa-Xtrkt6mUwGjNYIzE0>

## Load data into Google Cloud Storage

In [3]:
storage_client = storage.Client.from_service_account_json(SERVICE_ACCOUNT_FILE)

In [4]:
BUCKET_NAME = "geocourse-data-engineering"
bucket = storage_client.get_bucket(BUCKET_NAME)
print(f"Connected to bucket: {bucket.name}")

Connected to bucket: geocourse-data-engineering


In [5]:
def upload_to_gcs(bucket_name, source_file_name, destination_blob_name):
  bucket = storage_client.bucket(bucket_name)
  blob = bucket.blob(destination_blob_name)
  blob.upload_from_filename(source_file_name)
  print(f"File {source_file_name} uploaded to {destination_blob_name}.")

In [9]:
file_path_gpkg

'../../transform/output.gpkg'

In [10]:
upload_to_gcs(BUCKET_NAME, file_path_gpkg, f"geospatial_data/{now("gpkg")}")

File ../../transform/output.gpkg uploaded to geospatial_data/2025-03-09_09-49-54.gpkg.


In [11]:
file_path_tiff

'../../transform/georeferenced.tif'

In [12]:
upload_to_gcs(BUCKET_NAME, file_path_tiff, f"geospatial_data/{now("tif")}")

File ../../transform/georeferenced.tif uploaded to geospatial_data/2025-03-09_09-51-02.tif.
